In [0]:
from pyspark.sql import functions as F

pares = spark.table("mvp_engdados.silver.episodes_imdb_matched")

pares_validos = pares.filter(
    F.col("us_viewers").isNotNull() &
    F.col("imdb_rating").isNotNull()
)

correlacao_por_serie = (
    pares_validos
    .groupBy("series_id")
    .agg(
        F.count("*").alias("episode_count"),
        F.round(F.corr("us_viewers", "imdb_rating"), 3)
         .alias("pearson_correlation"),
        F.round(F.avg("us_viewers"), 0).alias("avg_us_viewers"),
        F.round(F.avg("imdb_rating"), 2).alias("avg_imdb_rating")
    )
    .orderBy("series_id")
)

display(correlacao_por_serie)

series_id,episode_count,pearson_correlation,avg_us_viewers,avg_imdb_rating
big_bang_theory,279,-0.193,1.4646308E7,7.82
breaking_bad,57,0.54,2324386.0,9.04
brooklyn_99,153,0.007,2620719.0,8.13
friends,234,0.276,2.5159444E7,8.42
game_of_thrones,73,-0.431,6447808.0,8.75
how_i_met_your_mother,208,0.11,9044183.0,8.14
lost,117,-0.002,1.4537607E7,8.53
the_office,175,0.511,7117086.0,8.16
the_walking_dead,177,0.28,8137966.0,7.92


In [0]:
(
    correlacao_por_serie.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.audience_rating_correlation")
)

In [0]:
series_gold = spark.table("mvp_engdados.gold.series_metrics")
temporadas_gold = spark.table("mvp_engdados.gold.season_metrics")
correlacao_gold = spark.table(
    "mvp_engdados.gold.audience_rating_correlation"
)

display(
    series_gold
    .select(
        "series_id",
        "first_air_date",
        "last_air_date",
        "duration_days",
        "duration_years"
    )
    .orderBy(F.desc("duration_days"))
)

display(
    series_gold
    .select("series_id", "episode_count")
    .orderBy(F.desc("episode_count"))
)

display(
    series_gold
    .select("series_id", "avg_imdb_rating", "imdb_episode_count")
    .orderBy(F.desc("avg_imdb_rating"))
)

series_id,first_air_date,last_air_date,duration_days,duration_years
the_walking_dead,2010-10-31,2022-11-20,4403,12.05
big_bang_theory,2007-09-24,2019-05-16,4252,11.64
friends,1994-09-22,2004-05-06,3514,9.62
how_i_met_your_mother,2005-09-19,2014-03-31,3115,8.53
the_office,2005-03-24,2013-05-16,2975,8.15
game_of_thrones,2011-04-17,2019-05-19,2954,8.09
brooklyn_99,2013-09-17,2021-09-16,2921,8.0
breaking_bad,2008-01-20,2013-09-29,2079,5.69
lost,2004-09-22,2010-05-23,2069,5.66


series_id,episode_count
big_bang_theory,279
friends,236
how_i_met_your_mother,208
the_office,201
the_walking_dead,177
brooklyn_99,153
lost,121
game_of_thrones,73
breaking_bad,62


series_id,avg_imdb_rating,imdb_episode_count
breaking_bad,9.03,62
game_of_thrones,8.75,73
lost,8.55,119
friends,8.42,235
the_office,8.22,188
how_i_met_your_mother,8.14,208
brooklyn_99,8.13,153
the_walking_dead,7.92,177
big_bang_theory,7.82,279


In [0]:
display(
    temporadas_gold
    .select(
        "series_id",
        "season",
        "episode_count",
        "avg_imdb_rating",
        "avg_us_viewers"
    )
    .orderBy("series_id", "season")
)

display(
    correlacao_gold
    .orderBy(F.desc("pearson_correlation"))
)

series_id,season,episode_count,avg_imdb_rating,avg_us_viewers
big_bang_theory,1,17,8.14,8443529.0
big_bang_theory,2,23,8.24,1.0142609E7
big_bang_theory,3,23,8.26,1.4118696E7
big_bang_theory,4,24,8.11,1.2558333E7
big_bang_theory,5,24,8.0,1.488125E7
big_bang_theory,6,24,8.0,1.6799167E7
big_bang_theory,7,24,7.9,1.7695E7
big_bang_theory,8,24,7.49,1.74975E7
big_bang_theory,9,24,7.58,1.770875E7
big_bang_theory,10,24,7.41,1.6670833E7


series_id,episode_count,pearson_correlation,avg_us_viewers,avg_imdb_rating
breaking_bad,57,0.54,2324386.0,9.04
the_office,175,0.511,7117086.0,8.16
the_walking_dead,177,0.28,8137966.0,7.92
friends,234,0.276,2.5159444E7,8.42
how_i_met_your_mother,208,0.11,9044183.0,8.14
brooklyn_99,153,0.007,2620719.0,8.13
lost,117,-0.002,1.4537607E7,8.53
big_bang_theory,279,-0.193,1.4646308E7,7.82
game_of_thrones,73,-0.431,6447808.0,8.75


In [0]:
display(
    spark.table("mvp_engdados.gold.season_metrics")
    .select(
        "series_id",
        "season",
        "episode_count",
        "imdb_episode_count",
        "avg_imdb_rating",
        "avg_us_viewers"
    )
    .orderBy("series_id", "season")
)

series_id,season,episode_count,imdb_episode_count,avg_imdb_rating,avg_us_viewers
big_bang_theory,1,17,17,8.14,8443529.0
big_bang_theory,2,23,23,8.24,1.0142609E7
big_bang_theory,3,23,23,8.26,1.4118696E7
big_bang_theory,4,24,24,8.11,1.2558333E7
big_bang_theory,5,24,24,8.0,1.488125E7
big_bang_theory,6,24,24,8.0,1.6799167E7
big_bang_theory,7,24,24,7.9,1.7695E7
big_bang_theory,8,24,24,7.49,1.74975E7
big_bang_theory,9,24,24,7.58,1.770875E7
big_bang_theory,10,24,24,7.41,1.6670833E7


In [0]:
display(
    spark.table("mvp_engdados.gold.series_metrics")
    .select("series_id", "duration_years")
    .orderBy(F.desc("duration_years"))
)

series_id,duration_years
the_walking_dead,12.05
big_bang_theory,11.64
friends,9.62
how_i_met_your_mother,8.53
the_office,8.15
game_of_thrones,8.09
brooklyn_99,8.0
breaking_bad,5.69
lost,5.66


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.table("mvp_engdados.gold.series_metrics")
    .select("series_id", "episode_count")
    .orderBy(F.desc("episode_count"))
)

series_id,episode_count
big_bang_theory,279
friends,236
how_i_met_your_mother,208
the_office,201
the_walking_dead,177
brooklyn_99,153
lost,121
game_of_thrones,73
breaking_bad,62


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.table("mvp_engdados.gold.series_metrics")
    .select("series_id", "avg_imdb_rating")
    .orderBy(F.desc("avg_imdb_rating"))
)

series_id,avg_imdb_rating
breaking_bad,9.03
game_of_thrones,8.75
lost,8.55
friends,8.42
the_office,8.22
how_i_met_your_mother,8.14
brooklyn_99,8.13
the_walking_dead,7.92
big_bang_theory,7.82


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.table("mvp_engdados.gold.season_metrics")
    .select("series_id", "season", "avg_imdb_rating")
    .orderBy("series_id", "season")
)

series_id,season,avg_imdb_rating
big_bang_theory,1,8.14
big_bang_theory,2,8.24
big_bang_theory,3,8.26
big_bang_theory,4,8.11
big_bang_theory,5,8.0
big_bang_theory,6,8.0
big_bang_theory,7,7.9
big_bang_theory,8,7.49
big_bang_theory,9,7.58
big_bang_theory,10,7.41


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.table("mvp_engdados.gold.season_metrics")
    .select("series_id", "season", "avg_us_viewers")
    .orderBy("series_id", "season")
)

series_id,season,avg_us_viewers
big_bang_theory,1,8443529.0
big_bang_theory,2,1.0142609E7
big_bang_theory,3,1.4118696E7
big_bang_theory,4,1.2558333E7
big_bang_theory,5,1.488125E7
big_bang_theory,6,1.6799167E7
big_bang_theory,7,1.7695E7
big_bang_theory,8,1.74975E7
big_bang_theory,9,1.770875E7
big_bang_theory,10,1.6670833E7


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.table("mvp_engdados.gold.audience_rating_correlation")
    .select("series_id", "pearson_correlation")
    .orderBy("pearson_correlation")
)

series_id,pearson_correlation
game_of_thrones,-0.431
big_bang_theory,-0.193
lost,-0.002
brooklyn_99,0.007
how_i_met_your_mother,0.11
friends,0.276
the_walking_dead,0.28
the_office,0.511
breaking_bad,0.54


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

temporadas = spark.table("mvp_engdados.gold.season_metrics")

janela = Window.partitionBy("series_id").orderBy("season")

variacao_audiencia = (
    temporadas
    .select("series_id", "season", "avg_us_viewers")
    .withColumn(
        "previous_season_viewers",
        F.lag("avg_us_viewers").over(janela)
    )
    .withColumn(
        "audience_change_pct",
        F.when(
            F.col("previous_season_viewers").isNotNull() &
            (F.col("previous_season_viewers") != 0),
            F.round(
                (
                    (F.col("avg_us_viewers") -
                     F.col("previous_season_viewers")) /
                    F.col("previous_season_viewers")
                ) * 100,
                2
            )
        )
    )
)

display(
    variacao_audiencia
    .orderBy("series_id", "season")
)

series_id,season,avg_us_viewers,previous_season_viewers,audience_change_pct
big_bang_theory,1,8443529.0,null,null
big_bang_theory,2,1.0142609E7,8443529.0,20.12
big_bang_theory,3,1.4118696E7,1.0142609E7,39.2
big_bang_theory,4,1.2558333E7,1.4118696E7,-11.05
big_bang_theory,5,1.488125E7,1.2558333E7,18.5
big_bang_theory,6,1.6799167E7,1.488125E7,12.89
big_bang_theory,7,1.7695E7,1.6799167E7,5.33
big_bang_theory,8,1.74975E7,1.7695E7,-1.12
big_bang_theory,9,1.770875E7,1.74975E7,1.21
big_bang_theory,10,1.6670833E7,1.770875E7,-5.86


In [0]:
(
    variacao_audiencia.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.audience_season_change")
)

In [0]:
imdb = spark.table("mvp_engdados.silver.imdb")

imdb_para_analise = imdb.filter(
    ~(
        (F.col("series_id") == "big_bang_theory") &
        (F.lower(F.trim("title")) == "unaired pilot")
    )
)

consistencia_notas = (
    imdb_para_analise
    .groupBy("series_id")
    .agg(
        F.count("*").alias("imdb_episode_count"),
        F.round(F.avg("imdb_rating"), 2).alias("avg_imdb_rating"),
        F.round(F.stddev_samp("imdb_rating"), 2).alias("rating_stddev"),
        F.round(
            F.percentile_approx("imdb_rating", 0.5),
            2
        ).alias("median_imdb_rating"),
        F.min("imdb_rating").alias("min_imdb_rating"),
        F.max("imdb_rating").alias("max_imdb_rating")
    )
    .orderBy("rating_stddev")
)

display(consistencia_notas)

series_id,imdb_episode_count,avg_imdb_rating,rating_stddev,median_imdb_rating,min_imdb_rating,max_imdb_rating
friends,235,8.42,0.41,8.4,7.1,9.7
lost,119,8.55,0.47,8.5,7.2,9.7
big_bang_theory,279,7.82,0.5,7.8,6.8,9.5
breaking_bad,62,9.03,0.51,8.9,7.9,10.0
how_i_met_your_mother,208,8.14,0.58,8.1,5.5,9.5
brooklyn_99,153,8.13,0.58,8.1,4.8,9.6
the_office,188,8.22,0.62,8.2,6.4,9.8
the_walking_dead,177,7.92,0.82,7.9,4.1,9.6
game_of_thrones,73,8.75,0.93,8.8,4.1,9.9


In [0]:
(
    consistencia_notas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.rating_consistency")
)

In [0]:
mudanca_audiencia = spark.table("mvp_engdados.gold.audience_season_change")
consistencia_notas = spark.table("mvp_engdados.gold.rating_consistency")

display(
    mudanca_audiencia
    .filter(F.col("audience_change_pct").isNotNull())
    .orderBy(F.desc(F.abs("audience_change_pct")))
)

display(
    consistencia_notas
    .orderBy("rating_stddev")
)

series_id,season,avg_us_viewers,previous_season_viewers,audience_change_pct
breaking_bad,5,4315625.0,1866923.0,131.16
the_walking_dead,3,1.075125E7,6911538.0,55.56
game_of_thrones,2,3795000.0,2515000.0,50.89
the_walking_dead,11,1690417.0,3036818.0,-44.34
big_bang_theory,3,1.4118696E7,1.0142609E7,39.2
the_walking_dead,10,3036818.0,4951875.0,-38.67
game_of_thrones,4,6846000.0,4966000.0,37.86
the_walking_dead,9,4951875.0,7817500.0,-36.66
game_of_thrones,7,1.0261429E7,7688000.0,33.47
the_walking_dead,2,6911538.0,5235000.0,32.03


Databricks visualization. Run in Databricks to view.

series_id,imdb_episode_count,avg_imdb_rating,rating_stddev,median_imdb_rating,min_imdb_rating,max_imdb_rating
friends,235,8.42,0.41,8.4,7.1,9.7
lost,119,8.55,0.47,8.5,7.2,9.7
big_bang_theory,279,7.82,0.5,7.8,6.8,9.5
breaking_bad,62,9.03,0.51,8.9,7.9,10.0
how_i_met_your_mother,208,8.14,0.58,8.1,5.5,9.5
brooklyn_99,153,8.13,0.58,8.1,4.8,9.6
the_office,188,8.22,0.62,8.2,6.4,9.8
the_walking_dead,177,7.92,0.82,7.9,4.1,9.6
game_of_thrones,73,8.75,0.93,8.8,4.1,9.9


Databricks visualization. Run in Databricks to view.